In [ ]:
import json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

ROOT    = Path('..')
REPORTS = ROOT / 'reports'

SEVERITY_COLORS = {
    'Critical':'#d62728','High':'#ff7f0e','Medium':'#f7c948',
    'Low':'#2ca02c','Negligible':'#aec7e8','Unknown':'#c7c7c7',
    'ERROR':'#d62728','WARNING':'#ff7f0e','INFO':'#aec7e8',
    'HIGH':'#d62728','MEDIUM':'#ff7f0e','LOW':'#2ca02c',
}

# ── Carga repos ───────────────────────────────────────────────────────────────
with open(REPORTS / 'repos.json') as f: repos_raw = json.load(f)
ORG = repos_raw['org']
df_repos = pd.DataFrame(repos_raw['repos'])

# ── Carga SBOMs ───────────────────────────────────────────────────────────────
with open(REPORTS / 'sbom_summary.json') as f: sbom_raw = json.load(f)
sbom_rows = []
for s in sbom_raw['sboms']:
    if 'error' in s: continue
    for eco, cnt in s.get('ecosystems',{}).items():
        sbom_rows.append({'repo':s['repo'],'ecosystem':eco,'components':cnt})
df_sbom = pd.DataFrame(sbom_rows) if sbom_rows else pd.DataFrame(columns=['repo','ecosystem','components'])
df_sbom_summary = pd.DataFrame([{'repo':s['repo'],'total_components':s.get('component_count',0)}
                                  for s in sbom_raw['sboms'] if 'error' not in s])

# ── Carga vulnerabilidades ────────────────────────────────────────────────────
with open(REPORTS / 'vulnerability_report.json') as f: vuln_raw = json.load(f)
results_vuln = [r for r in vuln_raw['results'] if 'error' not in r]
rows = []
for r in results_vuln:
    sc = r.get('severity_counts', {})
    rows.append({'repo':r['repo'],'total_vulns':r['total_vulnerabilities'],
                 'critical':sc.get('Critical',0),'high':sc.get('High',0),
                 'medium':sc.get('Medium',0),'low':sc.get('Low',0),
                 'negligible':sc.get('Negligible',0),'unknown':sc.get('Unknown',0),
                 'packages_affected':r.get('packages_affected_count',0),
                 'max_cvss':r.get('max_cvss_score',0.0),'fixable':r.get('fixable',0)})
df_vuln = pd.DataFrame(rows).sort_values('total_vulns',ascending=False).reset_index(drop=True)

total_vulns    = df_vuln['total_vulns'].sum()
total_critical = df_vuln['critical'].sum()
total_high     = df_vuln['high'].sum()
total_medium   = df_vuln['medium'].sum()
total_low      = df_vuln['low'].sum()
total_fixable  = df_vuln['fixable'].sum()
fix_pct        = (total_fixable / total_vulns * 100) if total_vulns > 0 else 0
pct = lambda n: f'{n/total_vulns*100:.1f}%' if total_vulns > 0 else 'N/A'

# ── Carga análisis de código ──────────────────────────────────────────────────
with open(REPORTS / 'code_analysis_report.json') as f: code_raw = json.load(f)
results_code = [r for r in code_raw['results'] if 'error' not in r]
code_rows = []
for r in results_code:
    sc = r.get('severity_counts', {})
    code_rows.append({'repo':r['repo'],'total_findings':r['total_findings'],
                      'error':sc.get('ERROR',0),'warning':sc.get('WARNING',0),
                      'info':sc.get('INFO',0),'categories':r.get('categories',{})})
df_code = pd.DataFrame(code_rows).sort_values('total_findings',ascending=False).reset_index(drop=True)

# ── Carga análisis CI/CD ──────────────────────────────────────────────────────
with open(REPORTS / 'cicd_report.json') as f: cicd_raw = json.load(f)
cicd_rows = []
for r in cicd_raw['results']:
    sc = r.get('severity_counts', {})
    cicd_rows.append({'repo':r['repo'],'workflow_count':r['workflow_count'],
                      'total_findings':r['total_findings'],
                      'high':sc.get('HIGH',0),'medium':sc.get('MEDIUM',0),
                      'low':sc.get('LOW',0),'info':sc.get('INFO',0)})
df_cicd = pd.DataFrame(cicd_rows).sort_values('total_findings',ascending=False).reset_index(drop=True)

display(Markdown(f'# Análisis de Seguridad — Organización `{ORG}`'))
display(Markdown(f'**Repositorios analizados:** {len(df_repos)} (top 5 por estrellas) &nbsp;|&nbsp; **Organización:** [{ORG}](https://github.com/{ORG})'))

In [ ]:
# ── Repositorios seleccionados ────────────────────────────────────────────────
display(Markdown('## Repositorios analizados'))
display(
    df_repos[['name','language','stargazers_count','forks_count','open_issues_count','size_kb']]
    .rename(columns={'name':'Repositorio','language':'Lenguaje','stargazers_count':'⭐ Estrellas',
                     'forks_count':'Forks','open_issues_count':'Issues abiertos','size_kb':'Tamaño (KB)'})
    .style.background_gradient(subset=['⭐ Estrellas'], cmap='YlOrRd')
    .format({'⭐ Estrellas':'{:,}','Forks':'{:,}','Tamaño (KB)':'{:,}'})
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Popularidad de repositorios', fontweight='bold')
colors = sns.color_palette('Blues_d', len(df_repos))
axes[0].barh(df_repos['name'][::-1], df_repos['stargazers_count'][::-1], color=colors)
axes[0].set_title('Estrellas en GitHub'); axes[0].set_xlabel('Estrellas')
for i, v in enumerate(df_repos['stargazers_count'][::-1]):
    axes[0].text(v + df_repos['stargazers_count'].max()*0.01, i, f'{v:,}', va='center', fontsize=9)

lang_counts = df_repos['language'].fillna('Unknown').value_counts()
axes[1].pie(lang_counts, labels=lang_counts.index, autopct='%1.0f%%',
            colors=sns.color_palette('pastel', len(lang_counts)), startangle=140)
axes[1].set_title('Lenguajes de programación')
plt.tight_layout(); plt.savefig(REPORTS/'fig_repos.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Dependencias ─────────────────────────────────────────────────────────────
display(Markdown('## Dependencias (SBOMs)'))
if not df_sbom.empty:
    eco_total = df_sbom.groupby('ecosystem')['components'].sum().sort_values(ascending=False)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Componentes por ecosistema', fontweight='bold')
    palette = sns.color_palette('tab10', len(eco_total))
    axes[0].bar(eco_total.index, eco_total.values, color=palette)
    axes[0].set_xlabel('Ecosistema'); axes[0].set_ylabel('Componentes')
    axes[0].tick_params(axis='x', rotation=45)
    top_n = min(8, len(eco_total))
    pie_data = eco_total.head(top_n).copy()
    if len(eco_total) > top_n: pie_data['otros'] = eco_total.iloc[top_n:].sum()
    axes[1].pie(pie_data, labels=pie_data.index, autopct='%1.1f%%',
                colors=sns.color_palette('pastel', len(pie_data)), startangle=140)
    axes[1].set_title('Proporción')
    plt.tight_layout(); plt.savefig(REPORTS/'fig_ecosystems.png',bbox_inches='tight'); plt.show()

top_comp = df_sbom_summary.sort_values('total_components', ascending=False)
plt.figure(figsize=(10, 4))
plt.barh(top_comp['repo'][::-1], top_comp['total_components'][::-1],
         color=sns.color_palette('Blues_d', len(top_comp)))
plt.title('Componentes por repositorio', fontweight='bold'); plt.xlabel('Componentes')
for i, v in enumerate(top_comp['total_components'][::-1]):
    plt.text(v + top_comp['total_components'].max()*0.01, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout(); plt.savefig(REPORTS/'fig_components.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Vulnerabilidades en dependencias ─────────────────────────────────────────
display(Markdown('## Vulnerabilidades en dependencias (Grype)'))
display(Markdown(f"""
| Métrica | Valor |
|---|---|
| Total vulnerabilidades | {total_vulns:,} |
| 🔴 Critical | {total_critical:,} ({pct(total_critical)}) |
| 🟠 High | {total_high:,} ({pct(total_high)}) |
| 🟡 Medium | {total_medium:,} ({pct(total_medium)}) |
| 🟢 Low | {total_low:,} ({pct(total_low)}) |
| Corregibles | {total_fixable:,} ({fix_pct:.1f}%) |
| CVSS máximo | {df_vuln['max_cvss'].max():.1f} |
"""))

sev_series = pd.Series({'Critical':total_critical,'High':total_high,'Medium':total_medium,
                         'Low':total_low,'Negligible':df_vuln['negligible'].sum()})
sev_series = sev_series[sev_series > 0]

if not sev_series.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Distribución de severidad — Dependencias', fontweight='bold')
    bar_colors = [SEVERITY_COLORS.get(s,'#999') for s in sev_series.index]
    axes[0].bar(sev_series.index, sev_series.values, color=bar_colors, edgecolor='white')
    axes[0].set_ylabel('Cantidad')
    for i, v in enumerate(sev_series.values):
        axes[0].text(i, v+sev_series.max()*0.01, f'{v:,}', ha='center', fontweight='bold')
    axes[1].pie(sev_series, labels=sev_series.index, colors=bar_colors, autopct='%1.1f%%', startangle=90)
    plt.tight_layout(); plt.savefig(REPORTS/'fig_severity.png',bbox_inches='tight'); plt.show()
else:
    display(Markdown('> ✅ No se encontraron vulnerabilidades en las dependencias.'))

# Stacked bar por repo
if total_vulns > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    bottom = np.zeros(len(df_vuln))
    for col, label in [('critical','Critical'),('high','High'),('medium','Medium'),('low','Low')]:
        vals = df_vuln[col].values
        if vals.sum() == 0: continue
        ax.barh(df_vuln['repo'][::-1], vals[::-1], left=bottom[::-1],
                label=label, color=SEVERITY_COLORS[label], edgecolor='white')
        bottom += vals
    ax.set_title('Vulnerabilidades por repositorio', fontweight='bold'); ax.set_xlabel('Vulnerabilidades')
    handles = [mpatches.Patch(color=SEVERITY_COLORS[s],label=s)
               for s in ['Critical','High','Medium','Low'] if df_vuln[s.lower()].sum()>0]
    if handles: ax.legend(handles=handles, loc='lower right')
    plt.tight_layout(); plt.savefig(REPORTS/'fig_vulns_repos.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Análisis estático de código (Semgrep) ─────────────────────────────────────
display(Markdown('## Análisis estático de código (Semgrep)'))

total_code_findings = df_code['total_findings'].sum()
display(Markdown(f'**Total findings:** {total_code_findings:,} | **ERROR:** {df_code["error"].sum():,} | **WARNING:** {df_code["warning"].sum():,} | **INFO:** {df_code["info"].sum():,}'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hallazgos de análisis estático (Semgrep)', fontweight='bold')

# Findings por repo
colors = sns.color_palette('Reds_d', len(df_code))
axes[0].barh(df_code['repo'][::-1], df_code['total_findings'][::-1], color=colors)
axes[0].set_title('Findings por repositorio'); axes[0].set_xlabel('Findings')
for i, v in enumerate(df_code['total_findings'][::-1]):
    axes[0].text(v + df_code['total_findings'].max()*0.01, i, str(v), va='center', fontsize=9)

# Distribución por severidad
sev_code = pd.Series({'ERROR':df_code['error'].sum(),'WARNING':df_code['warning'].sum(),'INFO':df_code['info'].sum()})
sev_code = sev_code[sev_code > 0]
if not sev_code.empty:
    axes[1].pie(sev_code, labels=sev_code.index,
                colors=[SEVERITY_COLORS.get(s,'#999') for s in sev_code.index],
                autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Proporción por severidad')
plt.tight_layout(); plt.savefig(REPORTS/'fig_code_analysis.png',bbox_inches='tight'); plt.show()

# Categorías de findings
all_cats = {}
for r in results_code:
    for cat, cnt in r.get('categories',{}).items():
        all_cats[cat] = all_cats.get(cat, 0) + cnt
if all_cats:
    cat_series = pd.Series(all_cats).sort_values(ascending=False).head(10)
    plt.figure(figsize=(12, 5))
    plt.barh(cat_series.index[::-1], cat_series.values[::-1], color=sns.color_palette('Oranges_d', len(cat_series)))
    plt.title('Top categorías de hallazgos (Semgrep)', fontweight='bold'); plt.xlabel('Cantidad')
    plt.tight_layout(); plt.savefig(REPORTS/'fig_code_categories.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Análisis CI/CD ────────────────────────────────────────────────────────────
display(Markdown('## Análisis de configuración CI/CD (GitHub Actions)'))

total_cicd_findings = df_cicd['total_findings'].sum()
total_wf = df_cicd['workflow_count'].sum()
display(Markdown(f'**Workflows analizados:** {total_wf} | **Total findings:** {total_cicd_findings:,} | **HIGH:** {df_cicd["high"].sum()} | **MEDIUM:** {df_cicd["medium"].sum()} | **LOW:** {df_cicd["low"].sum()}'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hallazgos en configuración CI/CD', fontweight='bold')

# Stacked por repo
bottom = np.zeros(len(df_cicd))
for col, label in [('high','HIGH'),('medium','MEDIUM'),('low','LOW'),('info','INFO')]:
    vals = df_cicd[col].values
    if vals.sum() == 0: continue
    axes[0].barh(df_cicd['repo'][::-1], vals[::-1], left=bottom[::-1],
                 label=label, color=SEVERITY_COLORS.get(label,'#999'), edgecolor='white')
    bottom += vals
axes[0].set_title('Findings por repositorio'); axes[0].set_xlabel('Findings')
handles = [mpatches.Patch(color=SEVERITY_COLORS[s],label=s)
           for s in ['HIGH','MEDIUM','LOW'] if df_cicd[s.lower()].sum()>0]
if handles: axes[0].legend(handles=handles)

# Tipos de problemas encontrados
rule_counts = {}
for r in cicd_raw['results']:
    for f in r.get('findings',[]):
        rid = f['rule_id']
        rule_counts[rid] = rule_counts.get(rid, 0) + 1
if rule_counts:
    rule_series = pd.Series(rule_counts).sort_values(ascending=False)
    rule_colors = []
    for rid in rule_series.index:
        sev = next((p['severity'] for p in __import__('importlib').import_module('builtins').__dict__.get('RISK_PATTERNS',[]) if p['id']==rid), 'LOW') if False else 'MEDIUM'
        rule_colors.append(SEVERITY_COLORS.get(sev,'#999'))
    axes[1].barh(rule_series.index[::-1], rule_series.values[::-1],
                 color=sns.color_palette('Purples_d', len(rule_series)))
    axes[1].set_title('Tipos de problemas CI/CD'); axes[1].set_xlabel('Ocurrencias')
plt.tight_layout(); plt.savefig(REPORTS/'fig_cicd.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Dataset consolidado ───────────────────────────────────────────────────────
display(Markdown('## Dataset consolidado'))

df_consolidated = df_repos[['name','language','stargazers_count']].copy()
df_consolidated = df_consolidated.merge(
    df_sbom_summary.rename(columns={'total_components':'componentes'}), left_on='name', right_on='repo', how='left').drop(columns='repo')
df_consolidated = df_consolidated.merge(
    df_vuln[['repo','total_vulns','critical','high','medium','low','max_cvss']],
    left_on='name', right_on='repo', how='left').drop(columns='repo')
df_consolidated = df_consolidated.merge(
    df_code[['repo','total_findings']].rename(columns={'total_findings':'code_findings'}),
    left_on='name', right_on='repo', how='left').drop(columns='repo')
df_consolidated = df_consolidated.merge(
    df_cicd[['repo','workflow_count','total_findings']].rename(columns={'total_findings':'cicd_findings'}),
    left_on='name', right_on='repo', how='left').drop(columns='repo')
df_consolidated = df_consolidated.fillna(0)

# Guardar dataset
df_consolidated.to_csv(REPORTS / 'dataset_consolidado.csv', index=False)

display(
    df_consolidated.rename(columns={
        'name':'Repositorio','language':'Lenguaje','stargazers_count':'⭐',
        'componentes':'Componentes','total_vulns':'Vulns','critical':'Crit',
        'high':'High','medium':'Med','low':'Low','max_cvss':'CVSS máx',
        'code_findings':'Findings código','workflow_count':'Workflows','cicd_findings':'Findings CI/CD'
    })
    .style.background_gradient(subset=['Vulns','Crit'], cmap='Reds')
    .background_gradient(subset=['Findings código'], cmap='Oranges')
    .background_gradient(subset=['Findings CI/CD'], cmap='Purples')
    .format({'⭐':'{:,.0f}','CVSS máx':'{:.1f}'})
)

In [ ]:
# ── Análisis cuantitativo: comparación entre dimensiones ─────────────────────
display(Markdown('## Análisis cuantitativo: comparación entre dimensiones'))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparación de hallazgos por dimensión y repositorio', fontweight='bold')

repos_order = df_consolidated['name'].tolist()
x = np.arange(len(repos_order))
width = 0.25

axes[0].bar(x, df_consolidated['total_vulns'], color='#d62728', label='Vulns dependencias')
axes[0].set_title('Vulnerabilidades en dependencias'); axes[0].set_ylabel('Cantidad')
axes[0].set_xticks(x); axes[0].set_xticklabels(repos_order, rotation=30, ha='right')

axes[1].bar(x, df_consolidated['code_findings'], color='#ff7f0e', label='Findings código')
axes[1].set_title('Findings en código fuente')
axes[1].set_xticks(x); axes[1].set_xticklabels(repos_order, rotation=30, ha='right')

axes[2].bar(x, df_consolidated['cicd_findings'], color='#9467bd', label='Findings CI/CD')
axes[2].set_title('Findings en CI/CD')
axes[2].set_xticks(x); axes[2].set_xticklabels(repos_order, rotation=30, ha='right')

plt.tight_layout(); plt.savefig(REPORTS/'fig_dimensions.png',bbox_inches='tight'); plt.show()

In [ ]:
# ── Análisis cualitativo ──────────────────────────────────────────────────────
display(Markdown('## Análisis cualitativo'))

# Repo con más riesgo overall
df_consolidated['risk_score'] = (
    df_consolidated['critical']*10 + df_consolidated['high']*5 +
    df_consolidated['medium']*2 + df_consolidated['low'] +
    df_consolidated['code_findings']*0.5 + df_consolidated['cicd_findings']*3
)
top_risk = df_consolidated.sort_values('risk_score', ascending=False).iloc[0]
most_deps = df_consolidated.sort_values('componentes', ascending=False).iloc[0]
most_code = df_consolidated.sort_values('code_findings', ascending=False).iloc[0]
most_cicd = df_consolidated.sort_values('cicd_findings', ascending=False).iloc[0]

# Patrones detectados en CI/CD
cicd_rule_counts = {}
for r in cicd_raw['results']:
    for f in r.get('findings',[]):
        rid = f['rule_id']
        cicd_rule_counts[rid] = cicd_rule_counts.get(rid, 0) + 1
top_cicd_issue = max(cicd_rule_counts, key=cicd_rule_counts.get) if cicd_rule_counts else 'N/A'

eco_top = df_sbom.groupby('ecosystem')['components'].sum().sort_values(ascending=False)
eco_top3 = ', '.join(eco_top.head(3).index.tolist()) if not eco_top.empty else 'N/A'

display(Markdown(f"""
### Patrones identificados

**1. Superficie de ataque en dependencias**  
Los repositorios analizados acumulan {df_sbom_summary['total_components'].sum():,} componentes de terceros distribuidos en los ecosistemas {eco_top3}. 
Esto representa una superficie de ataque significativa, consistente con el caso Apache Maven (2025), donde 
la distribución de artefactos manipulados en repositorios proxy afectó a proyectos que confían en dependencias externas sin verificación de integridad.

**2. Vulnerabilidades conocidas sin parchear**  
Se detectaron {total_vulns:,} vulnerabilidades en dependencias, de las cuales {total_critical:,} son críticas y {total_high:,} son de severidad alta. 
El {fix_pct:.1f}% tiene fix disponible, lo que sugiere que una parte importante del riesgo podría mitigarse con actualizaciones de dependencias.
El repositorio `{top_risk['name']}` concentra el mayor riesgo global.

**3. Problemas en código fuente**  
Semgrep identificó {df_code['total_findings'].sum():,} hallazgos en el código fuente. 
El repositorio `{most_code['name']}` presenta la mayor cantidad ({int(most_code['code_findings']):,} findings). 
Estos hallazgos incluyen malas prácticas de seguridad, exposición potencial de datos y patrones asociados al OWASP Top Ten.

**4. Configuraciones riesgosas en CI/CD**  
Se encontraron {df_cicd['total_findings'].sum():,} configuraciones riesgosas en los workflows de GitHub Actions. 
El problema más frecuente fue `{top_cicd_issue}`. 
Esto es directamente comparable con el caso `tj-actions/changed-files` (2025), donde la explotación de workflows de GitHub Actions permitió la filtración de secrets en pipelines de CI/CD comprometidos.

**5. Relación con casos de referencia**  
Los hallazgos de esta organización presentan similitudes estructurales con varios casos de la tabla inicial:
- Las vulnerabilidades en dependencias replican el patrón del caso **Apache Maven** y la campaña **Shai-Hulud**.
- Las configuraciones inseguras en workflows reflejan la problemática del ecosistema **GitHub Actions**.
- Los hallazgos de Semgrep en sanitización de entradas se relacionan con los casos de **ejecución remota de código** como Flowise y Gogs.

### Conclusión
Los problemas identificados no son exclusivos de esta organización, sino que reflejan condiciones más amplias del ecosistema de software: 
dependencia masiva de componentes de terceros, pipelines de CI/CD con configuraciones heredadas y deuda técnica acumulada en prácticas de seguridad. 
La mitigación requiere tanto acciones locales (actualizar dependencias, corregir workflows) como adoptar controles sistémicos como SBOMs en el pipeline de CI/CD y análisis estático automatizado.
"""))